# C10: City Swap for selected challenger winners

This notebook runs city-swap evaluation for a focused shortlist:

- best label smoothing challenger
- best r-drop challenger
- best class-balanced challenger
- best challenger GroupDRO
- best challenger focal model

Outputs are saved under `results/challengers_city_swap/c10_selected_family_city_swap/`.


In [ ]:
import importlib.util
import json
import sys
from pathlib import Path

import pandas as pd

CWD = Path.cwd().resolve()
NOTEBOOKS_DIR = CWD.parent if CWD.name == 'challengers' else CWD
EVAL_REPO = NOTEBOOKS_DIR.parent

PROJECT_ROOT_CANDIDATES = [
    EVAL_REPO.parent / 'my-repository',
    EVAL_REPO.parent / 'resume-screening',
    EVAL_REPO.parent / 'resume-screening' / 'resume-screening',
]
PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if p.exists()), PROJECT_ROOT_CANDIDATES[0])

CITY_SWAP_SCRIPT_CANDIDATES = [
    PROJECT_ROOT / 'city_swap_eval.py',
    PROJECT_ROOT.parent / 'city_swap_eval.py',
]
CITY_SWAP_SCRIPT = next((p for p in CITY_SWAP_SCRIPT_CANDIDATES if p.exists()), CITY_SWAP_SCRIPT_CANDIDATES[0])
TEST_CSV = PROJECT_ROOT / 'data' / 'processed' / 'test.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'challengers_city_swap' / 'c10_selected_family_city_swap'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('EVAL_REPO:', EVAL_REPO)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('CITY_SWAP_SCRIPT:', CITY_SWAP_SCRIPT)
print('TEST_CSV exists:', TEST_CSV.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)

if not CITY_SWAP_SCRIPT.exists():
    raise FileNotFoundError(f'city_swap_eval.py not found: {CITY_SWAP_SCRIPT}')
if not TEST_CSV.exists():
    raise FileNotFoundError(f'test.csv not found: {TEST_CSV}')

spec = importlib.util.spec_from_file_location('city_swap_eval_module', CITY_SWAP_SCRIPT)
city_swap_eval_module = importlib.util.module_from_spec(spec)
sys.modules['city_swap_eval_module'] = city_swap_eval_module
spec.loader.exec_module(city_swap_eval_module)
run_city_swap_eval = city_swap_eval_module.run_city_swap_eval


In [ ]:
SELECTED_MODELS = [
    {
        'family': 'label_smoothing',
        'model_name': 'label_smoothing_eps01_2ep',
        'path_candidates': [
            PROJECT_ROOT / 'notebooks' / 'models' / 'challengers' / 'label_smoothing_eps01_2ep',
            PROJECT_ROOT / 'models' / 'challengers' / 'label_smoothing_eps01_2ep',
        ],
    },
    {
        'family': 'rdrop',
        'model_name': 'rdrop_alpha10_2ep',
        'path_candidates': [
            PROJECT_ROOT / 'notebooks' / 'models' / 'challengers' / 'rdrop_alpha10_2ep',
            PROJECT_ROOT / 'models' / 'challengers' / 'rdrop_alpha10_2ep',
        ],
    },
    {
        'family': 'class_balanced',
        'model_name': 'class_balanced_cbce_beta099_2ep',
        'path_candidates': [
            PROJECT_ROOT / 'notebooks' / 'models' / 'challengers' / 'class_balanced_cbce_beta099_2ep',
            PROJECT_ROOT / 'models' / 'challengers' / 'class_balanced_cbce_beta099_2ep',
        ],
    },
    {
        'family': 'groupdro',
        'model_name': 'bert_gdro_eta005_2ep',
        'path_candidates': [
            PROJECT_ROOT / 'models' / 'challengers' / 'bert_gdro_eta005_2ep',
            PROJECT_ROOT / 'notebooks' / 'models' / 'challengers' / 'bert_gdro_eta005_2ep',
        ],
    },
    {
        'family': 'focal',
        'model_name': 'focal_gamma1_2ep',
        'path_candidates': [
            PROJECT_ROOT / 'notebooks' / 'models' / 'challengers' / 'focal_gamma1_2ep',
            PROJECT_ROOT / 'models' / 'challengers' / 'focal_gamma1_2ep',
        ],
    },
]

SWAP_CITIES = ['Москва', 'Екатеринбург', 'Новосибирск', 'Краснодар', 'Воронеж']
BATCH_SIZE = 32

def resolve_model_path(candidates):
    return next((Path(p) for p in candidates if Path(p).exists()), None)

resolved_rows = []
for item in SELECTED_MODELS:
    resolved_path = resolve_model_path(item['path_candidates'])
    resolved_rows.append({
        'family': item['family'],
        'model_name': item['model_name'],
        'resolved_model_path': str(resolved_path) if resolved_path else '',
        'exists': bool(resolved_path),
    })

resolved_df = pd.DataFrame(resolved_rows)
resolved_df


In [ ]:
summary_rows = []

for item in SELECTED_MODELS:
    model_name = item['model_name']
    family = item['family']
    model_path = resolve_model_path(item['path_candidates'])
    model_output_dir = OUTPUT_DIR / model_name
    summary_json = model_output_dir / 'city_swap_summary.json'

    row = {
        'family': family,
        'model_name': model_name,
        'model_path': str(model_path) if model_path else '',
        'output_dir': str(model_output_dir),
        'accuracy': None,
        'macro_f1': None,
        'overall_flip_rate': None,
        'status': '',
        'error': '',
    }

    if model_path is None:
        row['status'] = 'missing_model_dir'
        row['error'] = 'No matching model directory found in current workspace'
        summary_rows.append(row)
        continue

    try:
        print('\n' + '=' * 80)
        print('Running city swap for', model_name)
        print('Model path:', model_path)
        res = run_city_swap_eval(
            model_path=str(model_path),
            test_csv=str(TEST_CSV),
            swap_cities=SWAP_CITIES,
            output_dir=str(model_output_dir),
            batch_size=BATCH_SIZE,
        )
        row['accuracy'] = float(res.get('accuracy'))
        row['macro_f1'] = float(res.get('macro_f1'))
        row['overall_flip_rate'] = float(res.get('overall_flip_rate'))
        row['status'] = 'ok'

        if summary_json.exists():
            saved = json.loads(summary_json.read_text())
            for city in SWAP_CITIES:
                row[f'flip_{city}'] = saved.get('per_swap_city', {}).get(city, {}).get('flip_rate')
    except Exception as exc:
        row['status'] = 'error'
        row['error'] = f'{type(exc).__name__}: {exc}'

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'c10_selected_family_city_swap_summary.csv', index=False)
summary_df


In [ ]:
lines = ['=== C10 selected family city-swap summary ===']
for _, row in summary_df.iterrows():
    lines.extend([
        f"family: {row['family']}",
        f"model_name: {row['model_name']}",
        f"status: {row['status']}",
        f"accuracy: {row['accuracy']}",
        f"macro_f1: {row['macro_f1']}",
        f"overall_flip_rate: {row['overall_flip_rate']}",
        f"model_path: {row['model_path']}",
        f"error: {row['error'] if row['error'] else '-'}",
        '-' * 60,
    ])

report_text = '\n'.join(lines)
(OUTPUT_DIR / 'c10_selected_family_city_swap_report.txt').write_text(report_text)
print(report_text)
